# --- SECTION: INTRODUCTION ---
🥉 ElectroFlow: Building the Bronze Layer (Learning Edition)
In this notebook, we'll build the ingestion logic from scratch.
Follow the instructions in the comments and fill in the code!

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import uuid
import os

In [0]:
# We use os.getcwd() to find our position in the Workspace.
# We then strip the current folder '01_bronze' to find the Repo Root.
repo_root = os.getcwd().split("/01_bronze")[0]

#the ingestions function
def load_raw_to_bronze(source_path, target_table, file_format='csv'):
    print(f"Loading {source_path} to {target_table}")

    #read the raw data
    if file_format == "csv":
        df = spark.read.format("csv")\
            .option("header","true")\
            .option("inferschema","true")\
            .load(source_path)
    elif file_format == "json":
        df = spark.read.format('json')\
            .option("inferschema","true")\
            .option("multiline", "true")\
            .load(source_path) 

    #enrich the data
    df_enriched = df.select("*", F.col("_metadata.file_path").alias("_source_file_path")) \
                    .withColumn("_ingestion_timestamp", F.current_timestamp()) \
                    .withColumn("_ingestion_job_id", F.lit(str(uuid.uuid4())))

    #write to bronze table
    # Fix: Use a Unity Catalog volume location for Delta tables
    target_path = "/Volumes/dev/electroflow_pipeline/bronze_data/" + target_table
    df_enriched.write.format("delta")\
        .mode("overwrite")\
        .save(target_path)

    print (f"Finished ingestion to {target_table}")
    return df_enriched

# --- variables

landing_base = f"/00_landing/raw"

#files
bronze_customers = load_raw_to_bronze("/Workspace/Users/antonkooij@outlook.com/electroflow-pipeline/00_landing/raw/customers.csv", "bronze_customers")
bronze_products = load_raw_to_bronze("/Workspace/Users/antonkooij@outlook.com/electroflow-pipeline/00_landing/raw/products.csv", "bronze_products")
bronze_orders = load_raw_to_bronze("/Workspace/Users/antonkooij@outlook.com/electroflow-pipeline/00_landing/raw/orders.json", "bronze_orders", file_format="json")
bronze_payments = load_raw_to_bronze("/Workspace/Users/antonkooij@outlook.com/electroflow-pipeline/00_landing/raw/order_payments.csv", "bronze_order_payments")
bronze_coupons = load_raw_to_bronze("/Workspace/Users/antonkooij@outlook.com/electroflow-pipeline/00_landing/raw/coupons.csv", "bronze_coupons")
